In [1]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']
path = competition_download('ing-hubs-turkiye-datathon')

/home/osman/Desktop/Projects/ING_Datathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [3]:
customers["work_sector"] = customers["work_sector"].fillna(customers["work_type"])

In [4]:
customers["cust_age_month"] = (customers["age"] * 12)  - customers["tenure"]

In [5]:
def create_customer_features(df):
    """
    customer_history tablosundan cust_id bazlı özellikler üretir.
    - Kanal bazlı eksiklikler (nan) müşteriye özgü → 0 ile doldurulur.
    - Kullanım var/yok bilgisi (binary flag) eklenir.
    - İşlem sayısı ve tutarlar kendi aralarında tutarlı şekilde işlenir.
    
    Girdi:
        df (pd.DataFrame): Verilen sütunları içeren DataFrame
        
    Çıktı:
        features_df (pd.DataFrame): Her müşteri için türetilmiş özellikler
    """
    
    # Tarihi dönüştür
    df['date'] = pd.to_datetime(df['date'])
    df = df.copy()
    
    # Tüm sayısal sütunlar üzerinde müşteri bazlı "tamamen eksik mi?" kontrolü
    feature_cols = [
        'mobile_eft_all_cnt', 'mobile_eft_all_amt',
        'cc_transaction_all_cnt', 'cc_transaction_all_amt',
        'active_product_category_nbr'
    ]
    
    # Her müşteri için sütun bazlı tamamı nan mı? kontrolü
    def is_all_nan(group, col):
        return group[col].isna().all()
    
    features_list = []

    for cust_id, group in df.groupby('cust_id'):
        # Boş grup kontrolü
        if group.empty:
            continue
        
        # Kanal kullanım bilgileri
        uses_mobile_eft = not is_all_nan(group, 'mobile_eft_all_cnt')
        uses_cc = not is_all_nan(group, 'cc_transaction_all_cnt')
        
        # Eksikleri 0 yapmak mantıklı çünkü "kullanım yok"
        if not uses_mobile_eft:
            group['mobile_eft_all_cnt'] = -2
            group['mobile_eft_all_amt'] = -2
        if not uses_cc:
            group['cc_transaction_all_cnt'] = -2
            group['cc_transaction_all_amt'] = -2

        # active_product_category_nbr: bazı aylarda nan olabilir ama hep değil
        # Sadece tamamı nan ise 0 olur, değilse normal işleme devam
        if group['active_product_category_nbr'].isna().all():
            group['active_product_category_nbr'] = 0
        else:
            # Sadece nan olanları ileri/geri doldur, sonra 0
            group['active_product_category_nbr'] = group['active_product_category_nbr'].fillna(method='ffill').fillna(method='bfill').fillna(0)
        
        # Artık tüm NaN'ler temizlendi, sırala
        group = group.sort_values('date')
        
        # Özellik seti
        features = {
            'cust_id': cust_id,
            
            # -----------------------------
            # Kanal Kullanım Flag'leri
            # -----------------------------
            'uses_mobile_eft': int(uses_mobile_eft),
            'uses_cc': int(uses_cc),
            'uses_any_digital_channel': int(uses_mobile_eft or uses_cc),

            # -----------------------------
            # Mobil EFT (Hem sayı hem tutar)
            # -----------------------------
            'mobile_eft_cnt_mean': group['mobile_eft_all_cnt'].mean(),
            'mobile_eft_cnt_std': group['mobile_eft_all_cnt'].std() if len(group) > 1 else 0,
            'mobile_eft_cnt_min': group['mobile_eft_all_cnt'].min(),
            'mobile_eft_cnt_max': group['mobile_eft_all_cnt'].max(),
            'mobile_eft_cnt_last': group['mobile_eft_all_cnt'].iloc[-1],
            'mobile_eft_cnt_trend': np.polyfit(range(len(group)), group['mobile_eft_all_cnt'], 1)[0] 
                                   if len(group) > 1 else 0,

            'mobile_eft_amt_mean': group['mobile_eft_all_amt'].mean(),
            'mobile_eft_amt_std': group['mobile_eft_all_amt'].std() if len(group) > 1 else 0,
            'mobile_eft_amt_min': group['mobile_eft_all_amt'].min(),
            'mobile_eft_amt_max': group['mobile_eft_all_amt'].max(),
            'mobile_eft_amt_last': group['mobile_eft_all_amt'].iloc[-1],
            'mobile_eft_amt_trend': np.polyfit(range(len(group)), group['mobile_eft_all_amt'], 1)[0] 
                                   if len(group) > 1 else 0,

            # Ortalama işlem tutarı (sıfır bölünmesini önle)
            'mobile_eft_avg_amt_per_tx': (group['mobile_eft_all_amt'].sum() / group['mobile_eft_all_cnt'].sum() 
                                         if group['mobile_eft_all_cnt'].sum() > 0 else 0),

            # -----------------------------
            # Kredi Kartı (Sayı ve Tutar)
            # -----------------------------
            'cc_cnt_mean': group['cc_transaction_all_cnt'].mean(),
            'cc_cnt_std': group['cc_transaction_all_cnt'].std() if len(group) > 1 else 0,
            'cc_cnt_min': group['cc_transaction_all_cnt'].min(),
            'cc_cnt_max': group['cc_transaction_all_cnt'].max(),
            'cc_cnt_last': group['cc_transaction_all_cnt'].iloc[-1],
            'cc_cnt_trend': np.polyfit(range(len(group)), group['cc_transaction_all_cnt'], 1)[0] 
                           if len(group) > 1 else 0,

            'cc_amt_mean': group['cc_transaction_all_amt'].mean(),
            'cc_amt_std': group['cc_transaction_all_amt'].std() if len(group) > 1 else 0,
            'cc_amt_min': group['cc_transaction_all_amt'].min(),
            'cc_amt_max': group['cc_transaction_all_amt'].max(),
            'cc_amt_last': group['cc_transaction_all_amt'].iloc[-1],
            'cc_amt_trend': np.polyfit(range(len(group)), group['cc_transaction_all_amt'], 1)[0] 
                           if len(group) > 1 else 0,

            'cc_avg_amt_per_tx': (group['cc_transaction_all_amt'].sum() / group['cc_transaction_all_cnt'].sum() 
                                 if group['cc_transaction_all_cnt'].sum() > 0 else 0),

            # -----------------------------
            # Aktif Ürün Kategorisi
            # -----------------------------
            'active_product_mean': group['active_product_category_nbr'].mean(),
            'active_product_std': group['active_product_category_nbr'].std() if len(group) > 1 else 0,
            'active_product_min': group['active_product_category_nbr'].min(),
            'active_product_max': group['active_product_category_nbr'].max(),
            'active_product_last': group['active_product_category_nbr'].iloc[-1],
            'active_product_trend': np.polyfit(range(len(group)), group['active_product_category_nbr'], 1)[0] 
                                   if len(group) > 1 else 0,

            # Son 3 ay ortalama ürün kategorisi
            'active_product_recent_3m_mean': group['active_product_category_nbr'].tail(3).mean() if len(group) >= 3 else group['active_product_category_nbr'].mean(),

            # -----------------------------
            # Zaman Bilgisi
            # -----------------------------
            'tenure_months': len(group),
            'first_activity_month': group['date'].min(),
            'last_activity_month': group['date'].max(),

            # -----------------------------
            # Değişim (son 2 ay)
            # -----------------------------
            'cc_amt_change_1m': (group['cc_transaction_all_amt'].iloc[-1] - group['cc_transaction_all_amt'].iloc[-2]) 
                               if len(group) >= 2 else 0,
            'mobile_eft_amt_change_1m': (group['mobile_eft_all_amt'].iloc[-1] - group['mobile_eft_all_amt'].iloc[-2]) 
                                       if len(group) >= 2 else 0,
            'active_product_change_1m': (group['active_product_category_nbr'].iloc[-1] - group['active_product_category_nbr'].iloc[-2]) 
                                       if len(group) >= 2 else 0,

            # -----------------------------
            # Kullanım Aktiflik (son 2 ayda kullanım var mı?)
            # -----------------------------
            'mobile_eft_active_recent_2m': (group['mobile_eft_all_cnt'].tail(2).sum() > 0),
            'cc_active_recent_2m': (group['cc_transaction_all_cnt'].tail(2).sum() > 0),
        }

        features_list.append(features)

    # DataFrame'e çevir
    features_df = pd.DataFrame(features_list)

    # Tip dönüşümleri (bellek için opsiyonel)
    bool_cols = ['mobile_eft_active_recent_2m', 'cc_active_recent_2m']
    for col in bool_cols:
        features_df[col] = features_df[col].astype(int)
    
    # Son kontrol: NaN kontrolü
    features_df = features_df.fillna(-1)

    return features_df


In [ ]:
referance_data["ref_date"] = pd.to_datetime(referance_data["ref_date"])
referance_data_test["ref_date"] = pd.to_datetime(referance_data_test["ref_date"])
customer_history["date"] = pd.to_datetime(customer_history["date"])

def filter_history_before_ref_date(history_df, reference_df):
    """
    Her müşteri için, o müşterinin ref_date'inden SONRAKİ ay kayıtlarını eler.
    Bu filtre olmadan create_customer_features, churn kararının verildiği
    tarihten sonraki (henüz gerçekleşmemiş) işlem verisini kullanıyordu.
    """
    merged = history_df.merge(reference_df[["cust_id", "ref_date"]], on="cust_id", how="inner")
    return merged[merged["date"] <= merged["ref_date"]].drop(columns=["ref_date"])

train_history = filter_history_before_ref_date(customer_history, referance_data)
test_history = filter_history_before_ref_date(customer_history, referance_data_test)

In [ ]:
history_statics_train = create_customer_features(train_history)
history_statics_test = create_customer_features(test_history)

In [ ]:
new_customers_train = customers.merge(history_statics_train, "left", on="cust_id")
new_customers_test = customers.merge(history_statics_test, "left", on="cust_id")

train_data = new_customers_train.merge(referance_data, "right", on="cust_id")
test_data = new_customers_test.merge(referance_data_test, "right", on="cust_id")

In [ ]:
cat_cols = [col for col in train_data.columns if train_data[col].dtype == "object"]

In [ ]:
ohe_train_df = pd.get_dummies(train_data[cat_cols], drop_first=True).astype(int)
ohe_test_df = pd.get_dummies(test_data[cat_cols], drop_first=True).astype(int)

In [ ]:
new_train = pd.concat([train_data, ohe_train_df], axis=1).drop(cat_cols, axis=1)
new_test = pd.concat([test_data, ohe_test_df], axis=1).drop(cat_cols, axis=1)

In [ ]:
new_train = new_train.drop(["cust_id", "last_activity_month", "first_activity_month","ref_date"], axis=1)
new_test = new_test.drop(["cust_id", "last_activity_month", "first_activity_month","ref_date"], axis=1)

In [ ]:
new_train.to_csv("train_data.csv",index=False)
new_test.to_csv("test_data.csv", index=False)
sample_submission.to_csv("sample_submission.csv", index=False)